# 01 - Extract Firestore Data

This notebook connects to Firebase Firestore, reads all `sessions` documents, reads each session's `readings` subcollection, merges session metadata with sensor readings, and saves one raw CSV file for model building.

## Firebase credential setup

Do not hardcode Firebase private keys in this notebook.

Place your service account JSON file locally, for example:

`model building/secrets/firebase-service-account.json`

The `.gitignore` in this folder ignores JSON and `.env` files so private credentials are not committed.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
if cwd.name == "notebooks":
    BASE_DIR = cwd.parent
elif (cwd / "model building").exists():
    BASE_DIR = cwd / "model building"
else:
    BASE_DIR = cwd

sys.path.append(str(BASE_DIR / "src"))
RAW_OUTPUT_PATH = BASE_DIR / "data" / "raw" / "firestore_sensor_readings_raw.csv"
RAW_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Raw output path:", RAW_OUTPUT_PATH)

Base directory: /Users/vinushan/Documents/VAUED-Project/model building
Raw output path: /Users/vinushan/Documents/VAUED-Project/model building/data/raw/firestore_sensor_readings_raw.csv


## Load Firestore sessions and readings

The loader reads:

- `sessions/{session_id}`
- `sessions/{session_id}/readings/{reading_id}`

Optional fields such as `dust_level`, `sensor_status`, and `notes` are handled safely even if some readings do not contain them.

In [2]:
from firebase_loader import export_firestore_dataset

SERVICE_ACCOUNT_PATH = os.getenv(
    "FIREBASE_SERVICE_ACCOUNT_PATH",
    str(BASE_DIR / "secrets" / "firebase-service-account.json"),
)

if not Path(SERVICE_ACCOUNT_PATH).exists():
    raise FileNotFoundError(
        "Firebase service account JSON not found. Place it in "
        "model building/secrets/firebase-service-account.json or set "
        "FIREBASE_SERVICE_ACCOUNT_PATH."
    )

raw_df = export_firestore_dataset(
    service_account_path=SERVICE_ACCOUNT_PATH,
    output_csv_path=str(RAW_OUTPUT_PATH),
)

print("Rows exported:", len(raw_df))
raw_df.head()

Rows exported: 1171


,session_id,house_id,room_id,session_type,device_id,start_time,end_time,status,reading_interval_seconds,total_readings,...,reading_id,timestamp_ms,recorded_at,dust,air_quality,temperature,humidity,dust_level,sensor_status,notes
0,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_1a36383aaf1f,1776933925347,2026-04-23T08:45:28.150000+00:00,161.80,255.41,28.42,69.21,heavy,ok,None
1,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_1c0493cabf82,1776933941350,2026-04-23T08:45:42.253000+00:00,77.22,142.90,28.61,68.14,slight,ok,None
2,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_296f7b5ef1f1,1776933937349,2026-04-23T08:45:40.531000+00:00,103.80,167.56,28.53,66.58,heavy,ok,None
3,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_3b72f9cd8ad1,1776933982285,2026-04-23T08:46:22.961000+00:00,48.60,101.64,28.00,64.06,slight,ok,None
4,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,2026-04-23T08:45:17.411000+00:00,2026-04-23T08:46:24.906000+00:00,completed,4,15,...,reading_4e63c77ae9c4,1776933933345,2026-04-23T08:45:35.101000+00:00,131.52,208.98,28.61,67.09,heavy,ok,None


## Confirm exported columns

The final raw CSV should include session metadata, reading IDs, timestamps, and the main ML sensor values: `dust`, `air_quality`, `temperature`, and `humidity`.

In [3]:
expected_columns = [
    "session_id", "house_id", "room_id", "session_type", "device_id",
    "reading_id", "timestamp_ms", "recorded_at", "dust", "air_quality",
    "temperature", "humidity", "dust_level", "sensor_status", "notes",
]

missing = [column for column in expected_columns if column not in raw_df.columns]
print("Missing optional/expected columns:", missing)
print("Saved raw dataset to:", RAW_OUTPUT_PATH)
raw_df[raw_df.columns.intersection(expected_columns)].head()

Missing optional/expected columns: []
Saved raw dataset to: /Users/vinushan/Documents/VAUED-Project/model building/data/raw/firestore_sensor_readings_raw.csv


,session_id,house_id,room_id,session_type,device_id,reading_id,timestamp_ms,recorded_at,dust,air_quality,temperature,humidity,dust_level,sensor_status,notes
0,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,reading_1a36383aaf1f,1776933925347,2026-04-23T08:45:28.150000+00:00,161.80,255.41,28.42,69.21,heavy,ok,None
1,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,reading_1c0493cabf82,1776933941350,2026-04-23T08:45:42.253000+00:00,77.22,142.90,28.61,68.14,slight,ok,None
2,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,reading_296f7b5ef1f1,1776933937349,2026-04-23T08:45:40.531000+00:00,103.80,167.56,28.53,66.58,heavy,ok,None
3,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,reading_3b72f9cd8ad1,1776933982285,2026-04-23T08:46:22.961000+00:00,48.60,101.64,28.00,64.06,slight,ok,None
4,session_155a4e33a1e5,Frenten,Bed,after,esp32_all_sensors_01,reading_4e63c77ae9c4,1776933933345,2026-04-23T08:45:35.101000+00:00,131.52,208.98,28.61,67.09,heavy,ok,None
